In [ ]:
##SDR encoding method. 
####Tictactoe reference code : https://codelearn.io/sharing/day-ai-danh-tictactoe-voi-deep-learning
##DSQN reference code : https://github.com/mahmoudakl/dsrl
## wandb link : https://wandb.ai/kradeero-ohio-university/experiments/runs/9bu0y8po


import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import wandb
import os

class BaseModel:
    """Base class for reinforcement learning models"""
    def __init__(self, discount_factor, epsilon, e_min, e_max):
        """
        Initialize base parameters for reinforcement learning models
        
        Args:
            discount_factor (float): Discount factor for future rewards (gamma)
            epsilon (float): Initial exploration rate
            e_min (int): Minimum experiences before training starts
            e_max (int): Maximum experience replay buffer size
        """
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.e_max = e_max

class Tictactoe_v0:
    def __init__(self):
        self.board = [0] * 9
        self.wining_position = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                                [0, 3, 6], [1, 4, 7], [2, 5, 8],
                                [0, 4, 8], [6, 4, 2]]
        self.current_turn = 1
        self.player_mark = 1

    def reset(self, is_human_first):
        self.board = [0] * 9
        self.current_turn = 1
        self.player_mark = 1 if is_human_first else -1
        if not is_human_first:
            self.env_act()
        return self.board.copy()

    def check_win(self):
        for pst in self.wining_position:
            if str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]]) in ['111', '-1-1-1']:
                if self.current_turn == self.player_mark:
                    return 1, True #Player win
                return -1, True #AI win
        if 0 not in self.board:
            return 0, True #Draw
        return 0, False #Continue

    def env_act(self):
        action = random.choice([i for i in range(len(self.board)) if self.board[i] == 0])
        for pst in self.wining_position:
            com = str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]])
            if com.replace('0', '') == str(self.current_turn) * 2:
                if self.board[pst[0]] == 0:
                    action = pst[0]
                elif self.board[pst[1]] == 0:
                    action = pst[1]
                else:
                    action = pst[2]
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        return reward, done

    def step(self, action):
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        if not done:
            reward, done = self.env_act()
        return self.board.copy(), reward, done, None
    
    def render(self):
        """Visualize the board with X/O positions"""
        symbols = {1: 'X', -1: 'O', 0: ' '}
        print("\nCurrent Board:")
        for i in range(3):
            print(f" {symbols[self.board[i*3]]} | {symbols[self.board[i*3+1]]} | {symbols[self.board[i*3+2]]} ")
            if i < 2: print("-----------")
        print()

class EpsilonGreedy:
    def __init__(self, epsilon):
        self.epsilon = epsilon

    def perform(self, q_value, action_space: list = None):
        prob = np.random.sample()
        if prob <= self.epsilon:
            if action_space is None:
                return np.random.randint(len(q_value))
            return np.random.choice(action_space)
        else:
            if action_space is None:
                return np.argmax(q_value)
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]

    def decay(self, decay_value, lower_bound):
        self.epsilon = max(self.epsilon * decay_value, lower_bound)

class ExperienceReplay:
    def __init__(self, e_max: int):
        if e_max <= 0:
            raise ValueError('Invalid value for memory size')
        self.e_max = e_max
        self.memory = list()
        self.index = 0

    def add_experience(self, sample: list):
        if len(sample) != 5:
            raise Exception('Invalid sample')
        if len(self.memory) < self.e_max:
            self.memory.append(sample)
        else:
            self.memory[self.index] = sample
        self.index = (self.index + 1) % self.e_max

    def sample_experience(self, sample_size: int, cer_mode: bool):
        samples = random.sample(self.memory, sample_size)
        if cer_mode:
            samples[-1] = self.memory[self.index - 1]
        s_batch, a_batch, r_batch, ns_batch, done_batch = map(np.array, zip(*samples))
        return s_batch, a_batch, r_batch, ns_batch, done_batch

    def get_size(self):
        return len(self.memory)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        
        q_values = mem[-1]
        return q_values, mem_rec, spk

class DQN(BaseModel):
    """Deep Q-Network agent with Deep Spiking Neural Network"""
    def __init__(self, discount_factor: float, epsilon: float, e_min: int, e_max: int, dsnn_config: dict):
        super().__init__(discount_factor, epsilon, e_min, e_max)
        self.gamma = discount_factor
        self.epsilon_greedy = EpsilonGreedy(epsilon)
        self.e_min = e_min
        self.exp_replay = ExperienceReplay(e_max)
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.cache = []  # Initialize cache for storing experiences

        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        
        self.sdr_size = 100  # Size of SDR population
        self.sparsity = 0.1  # 10% active neurons
        self.seed = dsnn_config['seed']
        np.random.seed(self.seed)
        # Initialize random projection matrix for SDR
        self.projection_matrix = np.random.randn(9 * 3, self.sdr_size) * 0.1  # Map 9 cells * 3 states to SDR

    def population_encode(self, state):
        """Convert board state to Sparse Distributed Representation (SDR)"""
        encoded = torch.zeros(self.simulation_time, self.sdr_size, device=device)
        
        # One-hot encode the board state (9 cells, 3 states: X=1, O=-1, Empty=0)
        one_hot_state = np.zeros(9 * 3)
        for i, val in enumerate(state):
            if val == 1:
                one_hot_state[i*3] = 1  # X
            elif val == -1:
                one_hot_state[i*3 + 1] = 1  # O
            else:
                one_hot_state[i*3 + 2] = 1  # Empty
        
        # Project to SDR space
        sdr = np.dot(one_hot_state, self.projection_matrix)
        # Select top-k neurons to enforce sparsity (top 10%)
        k = int(self.sdr_size * self.sparsity)
        active_indices = np.argpartition(sdr, -k)[-k:]
        sdr_binary = np.zeros(self.sdr_size)
        sdr_binary[active_indices] = 1.0
        
        # Apply over simulation time with slight noise to ensure temporal dynamics
        for t in range(self.simulation_time):
            noise = np.random.normal(0, 0.01, self.sdr_size)
            noisy_sdr = sdr_binary + noise
            noisy_sdr = np.clip(noisy_sdr, 0, 1)
            encoded[t] = torch.tensor(noisy_sdr, device=device)
        
        return encoded.unsqueeze(0)

    def observe(self, state, action_space: list = None):
        """Get best action for given state (exploitation)"""
        with torch.no_grad():
            encoded_state = self.population_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()

        if action_space is not None:
            valid_q = [(q_values[a], a) for a in action_space]
            return max(valid_q, key=lambda x: x[0])[1]
        return np.argmax(q_values)

    def observe_on_training(self, state, action_space: list = None) -> int:
        """Get action with epsilon-greedy exploration"""
        with torch.no_grad():
            encoded_state = self.population_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()

        action = self.epsilon_greedy.perform(q_values, action_space)
        self.cache.extend([state, action])
        return action

    def take_reward(self, reward, next_state, done):
        """Store experience in replay buffer"""
        self.cache.extend([reward, next_state, done])
        self.exp_replay.add_experience(self.cache.copy())
        self.cache.clear()

    def train_network(self, sample_size: int, batch_size: int):
        """Train network with SDR-encoded experiences"""
        if self.exp_replay.get_size() < self.e_min:
            return None

        states, actions, rewards, next_states, dones = self.exp_replay.sample_experience(sample_size, cer_mode=False)
        
        state_batch = torch.stack([self.population_encode(s) for s in states]).squeeze(1)
        next_state_batch = torch.stack([self.population_encode(ns) for ns in next_states]).squeeze(1)
        
        action_batch = torch.LongTensor(actions).to(device)
        reward_batch = torch.FloatTensor(rewards).to(device)
        done_batch = torch.BoolTensor(dones).to(device)

        with torch.no_grad():
            next_q_values, _, _ = self.target_net(next_state_batch)
            max_next_q = next_q_values.max(1)[0]
            target_q = reward_batch + (1 - done_batch.float()) * self.gamma * max_next_q

        current_q, mem_rec, spk_rec = self.training_net(state_batch)
        current_q = current_q.gather(1, action_batch.unsqueeze(1)).squeeze(1)

        self.training_net.optimizer.zero_grad()
        loss = F.mse_loss(current_q, target_q)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

        return loss.item()

    def update_target_network(self):
        """Update target network with training network weights"""
        self.target_net.load_state_dict(self.training_net.state_dict())

    def save_model(self, filename):
        """Save model weights to file"""
        torch.save({
            'training_net': self.training_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'epsilon': self.epsilon_greedy.epsilon
        }, filename)

    def load_model(self, filename):
        """Load model weights from file"""
        checkpoint = torch.load(filename)
        self.training_net.load_state_dict(checkpoint['training_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.epsilon_greedy.epsilon = checkpoint['epsilon']

def print_spikes(spk_rec, timestep=-1):
    """Print spike activity for last timestep"""
    print("\nSpike Activity:")
    for layer_idx, layer_spikes in enumerate(spk_rec):
        if len(layer_spikes) > 0:
            spike_count = layer_spikes[timestep].sum().item()
            print(f"Layer {layer_idx+1}: {spike_count} spikes")

def print_all_spikes(spk_rec):
    for layer_idx, layer_spikes_list in enumerate(spk_rec):
        if not layer_spikes_list:
            print(f"[Debug] Layer {layer_idx+1} has no spike data.")
            continue
        
        print(f"\n[Debug] Layer {layer_idx+1} Spikes:")
        layer_spikes_tensor = torch.stack(layer_spikes_list, dim=0)
        print(layer_spikes_tensor)

def print_encoded_state(encoded_state, timesteps=1):
    """Print SDR encoding for first few timesteps"""
    print("\nSDR Encoding (All Timesteps):")
    encoded_np = encoded_state.squeeze(0).cpu().numpy()
    
    for t in range(encoded_np.shape[0]):
        print(f"\nTimestep {t+1}:")
        active_neurons = np.where(encoded_np[t] > 0)[0]
        print(f"Active Neurons: {active_neurons} (Count: {len(active_neurons)})")

def print_membrane_potentials(q_values, action):
    """Show output layer decision process"""
    print("\nOutput Membrane Potentials (Q-values):")
    q_np = q_values.detach().cpu().numpy().flatten()
    for i in range(9):
        print(f"Position {i}: {q_np[i]:.2f}")
    print(f"Selected Action: Position {action} (Q-value: {q_np[action]:.2f})")

dsnn_config = {
    'architecture': [100, 128, 128, 9],  # Updated input size for SDR
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.85,
    'weight_scale': 0.15,
    'batch_size': 32,
    'threshold': 0.1,
    'simulation_time': 5,
    'learning_rate': 0.0001,
    'reset_potential': 0.0
}

env = Tictactoe_v0()
agent = DQN(
    discount_factor=0.95,
    epsilon=1.0,
    e_min=1000,
    e_max=100000,
    dsnn_config=dsnn_config
)

agent.update_target_network()

num_episodes = 20001
batch_size = 32
epochs = 1
sample_size = 64

total_loss = 0
episode_count_for_avg_loss = 0
win_count = 0
loss_count = 0
draw_count = 0
total_spike_count = 0
episode_count_for_avg_spikes = 0

wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="sdr ttt")

for episode in range(num_episodes):
    state = env.reset(is_human_first=True)
    done = False
    episode_reward = 0

    while not done:
        action_space = [i for i, val in enumerate(state) if val == 0]
        action = agent.observe_on_training(state, action_space)
        next_state, reward, done, _ = env.step(action)
        episode_reward += reward
        
        if episode % 100 == 0:
            encoded_state = agent.population_encode(state)
            with torch.no_grad():
                q_values, mem_rec, spk_rec = agent.training_net(encoded_state)
                
            step_spike_count = 0
            for layer_spikes in spk_rec:
                if layer_spikes:
                    for t in range(len(layer_spikes)):
                        step_spike_count += layer_spikes[t].sum().item()
            total_spike_count += step_spike_count
            episode_count_for_avg_spikes += 1
            
        agent.take_reward(reward, next_state, done)

        if agent.exp_replay.get_size() > agent.e_min:
            loss = agent.train_network(sample_size, batch_size)
            if loss is not None:
                total_loss += loss
                episode_count_for_avg_loss += 1
            agent.update_target_network()

        state = next_state
    
    agent.epsilon_greedy.decay(decay_value=0.995, lower_bound=0.01)

    if episode_reward == 1:
        win_count += 1
    elif episode_reward == -1:
        loss_count += 1
    else:
        if done:
            draw_count += 1

    combined_rate = win_count + draw_count

    if episode % 100 == 0:
        avg_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
        avg_spike_count = total_spike_count / episode_count_for_avg_spikes if episode_count_for_avg_spikes > 0 else 0
        print(f"Episode Summary {episode}, Games Won: {win_count}, Games Lost: {loss_count}, Games Drawn: {draw_count}")
        print(f"Episode: {episode}, Avg Loss: {avg_loss:.4f}, Avg Spikes: {avg_spike_count:.2f}")

        wandb.log({
            "Episode": episode,
            "Win Rate": win_count,
            "Loss Rate": loss_count,
            "Draw Rate": draw_count,
            "Win + Draw Rate": combined_rate,
            "Average Loss": avg_loss,
            "Average Spikes": avg_spike_count
        })

        total_loss = 0.0
        episode_count_for_avg_loss = 0
        win_count = 0
        loss_count = 0
        draw_count = 0
        total_spike_count = 0
        episode_count_for_avg_spikes = 0

# Save model with proper path handling
filename = "../saved_models/sdr_ttt.pth"
agent.save_model(filename)
wandb.finish()

In [1]:
##Evaluation Code for SDR encoding method. 

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# DSNN
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        all_spikes = []
        ac_count = 0  # Count synaptic fan-out additions
        internal_mac_count = 0  # Count internal state update MACs
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            layer_spikes = []
            for l in range(len(self.weights)):
                # Synaptic operations (fan-out, counted as additions)
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                    num_spikes = torch.sum(input > 0).item()  # Binarize inputs
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                    num_spikes = torch.sum(spk[l-1][-1]).item()  # Count binary spikes
                ac_count += num_spikes * self.weights[l].size(1)  # ACs: additions for synaptic fan-out
                
                # Internal state updates (count MACs)
                num_neurons = self.weights[l].size(1)
                internal_mac_count += num_neurons * 2  # 1 mult (alpha * syn) + 1 add, 1 mult (beta * mem) + 1 add
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    # Additional MACs for reset (approximated as 2 per neuron if spiking)
                    internal_mac_count += num_neurons * 2 * torch.mean(spk_current).item()  # Approx. 2 MACs per spiking neuron
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                    layer_spikes.append(spk_current)
                else:
                    layer_spikes.append(torch.zeros_like(mem[l]))
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
                
            all_spikes.append(layer_spikes)
        
        q_values = mem[-1]
        return q_values, mem_rec, all_spikes, ac_count, internal_mac_count

# DSQN with SDR encoding
class DSQN:
    def __init__(self, discount_factor=0.95, epsilon=0.1, e_min=1000, e_max=100000, dsnn_config=None):
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.exp_replay = None
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.sdr_size = 100  # Size of SDR population
        self.sparsity = 0.1  # 10% active neurons
        self.seed = dsnn_config['seed']
        np.random.seed(self.seed)
        self.projection_matrix = np.random.randn(9 * 3, self.sdr_size) * 0.1  # Map 9 cells * 3 states to SDR

        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)

    def population_encode(self, state):
        encoded = torch.zeros(self.simulation_time, self.sdr_size, device=device)
        one_hot_state = np.zeros(9 * 3)
        for i, val in enumerate(state):
            if val == 1:
                one_hot_state[i*3] = 1  # X
            elif val == -1:
                one_hot_state[i*3 + 1] = 1  # O
            else:
                one_hot_state[i*3 + 2] = 1  # Empty
        
        sdr = np.dot(one_hot_state, self.projection_matrix)
        k = int(self.sdr_size * self.sparsity)
        active_indices = np.argpartition(sdr, -k)[-k:]
        sdr_binary = np.zeros(self.sdr_size)
        sdr_binary[active_indices] = 1.0
        
        for t in range(self.simulation_time):
            noise = np.random.normal(0, 0.01, self.sdr_size)
            noisy_sdr = sdr_binary + noise
            noisy_sdr = np.clip(noisy_sdr, 0, 1)
            encoded[t] = torch.tensor(noisy_sdr, device=device)
        
        return encoded.unsqueeze(0)

    def observe(self, state, action_space=None):
        with torch.no_grad():
            encoded_state = self.population_encode(state)
            q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
            if action_space is not None:
                valid_q = [(q_values[a], a) for a in action_space]
                action = max(valid_q, key=lambda x: x[0])[1]
            else:
                action = np.argmax(q_values)
            return action, mem_rec, all_spikes, ac_count, internal_mac_count

    def load_model(self, filename):
        try:
            checkpoint = torch.load(filename, map_location=device)
            self.training_net.load_state_dict(checkpoint['training_net'])
            self.target_net.load_state_dict(checkpoint['target_net'])
            self.epsilon = checkpoint['epsilon']
            self.training_net.eval()
            self.target_net.eval()
            print(f"DSQN model loaded from {filename}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DSQN model file {filename} not found")

# Analysis functions
def analyze_decision_stabilization(mem_rec, simulation_time):
    decision_times = []
    final_decision = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        stabilized = False
        for t in range(simulation_time):
            current_decision = torch.argmax(mem_rec[t], dim=1)[b]
            if current_decision == final_decision[b]:
                stable = True
                for t_next in range(t, simulation_time):
                    if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
                        stable = False
                        break
                if stable:
                    decision_times.append(t + 1)
                    stabilized = True
                    break
        if not stabilized:
            decision_times.append(simulation_time)
    return decision_times

def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
    total_spikes = 0
    total_neurons = sum(architecture[1:-1])
    spike_counts = []
    
    for t in range(len(all_spikes)):
        for l in range(len(all_spikes[t])):
            if l < len(architecture) - 1:
                spikes = all_spikes[t][l]
                spike_count = torch.sum(spikes).item()
                total_spikes += spike_count
                spike_counts.append(spike_count)
    
    sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
    return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count

# Utility functions
def print_board(board, move=None, agent=None):
    print("\nBoard state:")
    symbols = {1: 'X', -1: 'O', 0: '_'}
    grid_board = [['_' for _ in range(3)] for _ in range(3)]
    for i, val in enumerate(board):
        row, col = i // 3, i % 3
        grid_board[row][col] = symbols[val]
    for row in grid_board:
        print(" | ".join(row))
        print("-" * 9)
    if move and agent:
        print(f"{agent} played at position ({move[0]}, {move[1]})")

def game_over(board):
    winning_positions = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],
        [0, 3, 6], [1, 4, 7], [2, 5, 8],
        [0, 4, 8], [2, 4, 6]
    ]
    for pos in winning_positions:
        if board[pos[0]] == board[pos[1]] == board[pos[2]] != 0:
            return True, 1 if board[pos[0]] == 1 else -1
    if 0 not in board:
        return True, 0
    return False, None

def random_move(board):
    action_space = [i for i, val in enumerate(board) if val == 0]
    return random.choice(action_space)

# Play game
def play_game(agent, agent_first=True, visualize=False, collect_analysis=False):
    flat_board = [0] * 9
    grid_board = [['_' for _ in range(3)] for _ in range(3)]
    current_player = 'x'
    agent_symbol = 'X'
    random_symbol = 'O'

    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if collect_analysis else None

    if visualize:
        print("\n=== New Game ===")
        print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
        print_board(flat_board)

    while True:
        if current_player == 'x':
            # Agent plays X
            action_space = [i for i, val in enumerate(flat_board) if val == 0]
            if collect_analysis:
                action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(flat_board, action_space)
                total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                analysis_data['total_spikes'].append(total_spikes)
                analysis_data['sparsity'].append(sparsity)
                analysis_data['ac_counts'].append(ac_count)
                analysis_data['internal_mac_counts'].append(internal_mac_count)
                decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                analysis_data['decision_times'].extend(decision_times)
            else:
                action = agent.observe(flat_board, action_space)[0]
            flat_board[action] = 1
            row, col = action // 3, action % 3
            grid_board[row][col] = 'x'
            if visualize:
                print_board(flat_board, (row, col), f"Agent ({agent_symbol})")
        else:
            # Random plays O
            action = random_move(flat_board)
            flat_board[action] = -1
            row, col = action // 3, action % 3
            grid_board[row][col] = 'o'
            if visualize:
                print_board(flat_board, (row, col), f"Random ({random_symbol})")

        done, result = game_over(flat_board)
        if done:
            if visualize:
                if result == 0:
                    print("Game ended in a draw!")
                elif result == 1:
                    print(f"Agent ({agent_symbol}) wins!")
                else:
                    print(f"Random ({random_symbol}) wins!")
            if result == 0:
                return 'draw', analysis_data
            elif result == 1:
                return 'agent', analysis_data
            else:
                return 'random', analysis_data

        current_player = 'o' if current_player == 'x' else 'x'

# Test agent vs random
def test_agent_vs_random(agent, agent_name, num_games=100, visualize_all=False):
    results = {'agent': 0, 'random': 0, 'draw': 0}
    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    }

    # Agent goes first (as X)
    agent.load_model("../saved_models/sdr_ttt.pth")
    for i in range(num_games):
        visualize = visualize_all
        winner, game_analysis = play_game(agent, agent_first=True, visualize=visualize, collect_analysis=True)
        results[winner] += 1
        if game_analysis:
            analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
            analysis_data['sparsity'].extend(game_analysis['sparsity'])
            analysis_data['decision_times'].extend(game_analysis['decision_times'])
            analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
            analysis_data['mac_counts'].extend(game_analysis['mac_counts'])
            analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

    # Print game results
    print(f"\n{agent_name} vs. Random Agent - Final Results (Agent as X)")
    print("="*50)
    print(f"Wins ({agent_name}): {results['agent']}, Losses: {results['random']}, Draws: {results['draw']}")
    print(f"Win rate: {results['agent'] / num_games:.2%}, "
          f"Loss rate: {results['random'] / num_games:.2%}, "
          f"Draw rate: {results['draw'] / num_games:.2%}")

    # Print analysis results
    if analysis_data['total_spikes']:
        print(f"\nDSQN (SDR Encoding) Analysis Results")
        print("="*50)
        print(f"Average Total Spikes per Decision: {float(np.mean(analysis_data['total_spikes'])):.2f} "
              f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
        print(f"Average Sparsity per Decision: {float(np.mean(analysis_data['sparsity'])):.2%} "
              f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
        print(f"Average ACs per Decision (spike-triggered additions): {float(np.mean(analysis_data['ac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
        print(f"Average Internal State Update MACs per Decision: {float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")
        decision_time_counts = np.bincount(analysis_data['decision_times'], minlength=agent.simulation_time + 1)[1:]
        print(f"Decision Stabilization Times (over all decisions):")
        for t in range(agent.simulation_time):
            print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
                  f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")
        # Compute energy ratio and savings
        total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
        total_neurons = 256  # 128 + 128 (hidden layers)
        total_possible_spikes = total_neurons * agent.simulation_time  # 256 * 5
        f_r = total_spikes_avg / total_possible_spikes
        avg_ac = float(np.mean(analysis_data['ac_counts']))
        avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))
        ann_energy = 18688 * 31  # ANN MACs * E_MAC (E_AC = 1)
        snn_energy = avg_ac + avg_internal_mac * 31  # E_SNN = AC * E_AC + Internal_MAC * E_MAC
        energy_ratio_detailed = snn_energy / ann_energy
        energy_savings_detailed = (1 - energy_ratio_detailed) * 100
        energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
        energy_savings_simplified = (1 - energy_ratio_simplified) * 100
        print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): {energy_ratio_simplified:.4f}")
        print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
        print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): {energy_ratio_detailed:.4f}")
        print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")

if __name__ == "__main__":
    # Set random seeds for reproducibility
    random.seed(82)
    np.random.seed(82)
    torch.manual_seed(82)

    # DSQN configuration
    dsnn_config = {
        'architecture': [100, 128, 128, 9],  # Updated input size for SDR
        'seed': 82,
        'alpha': 0.9,
        'beta': 0.85,
        'weight_scale': 0.15,
        'batch_size': 32,
        'threshold': 0.1,
        'simulation_time': 5,
        'learning_rate': 0.0001,
        'reset_potential': 0.0
    }

    # Initialize DSQN
    dsqn_agent = DSQN(dsnn_config=dsnn_config)

    print("\nTesting DSQN (SDR Encoding) vs. Random Agent (Agent as X)")
    test_agent_vs_random(dsqn_agent, "DSQN", num_games=100, visualize_all=False)


Testing DSQN (SDR Encoding) vs. Random Agent (Agent as X)
DSQN model loaded from ../saved_models/sdr_ttt.pth

DSQN vs. Random Agent - Final Results (Agent as X)
Wins (DSQN): 72, Losses: 16, Draws: 12
Win rate: 72.00%, Loss rate: 16.00%, Draw rate: 12.00%

DSQN (SDR Encoding) Analysis Results
Average Total Spikes per Decision: 244.71 (Std: 19.87)
Average Sparsity per Decision: 80.88% (Std: 1.55%)
Average ACs per Decision (spike-triggered additions): 50792.84 (Std: 2141.77)
Average Internal State Update MACs per Decision: 3139.42 (Std: 39.74)
Decision Stabilization Times (over all decisions):
  Time Step 1: 88 decisions (20.90%)
  Time Step 2: 209 decisions (49.64%)
  Time Step 3: 56 decisions (13.30%)
  Time Step 4: 38 decisions (9.03%)
  Time Step 5: 30 decisions (7.13%)
Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): 0.0308
Simplified Energy Savings: 96.9%
Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): 0.2557
Detailed Energy Savings: 74.4%


In [1]:
# ## Evaluation Code for SDR encoding method (single-seed, projection fixed to 42)

# import random
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# import os

# # Device configuration
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# # ================================================================
# # Surrogate Gradient Spike Function
# # ================================================================
# class SurrGradSpike(torch.autograd.Function):
#     scale = 100.0

#     @staticmethod
#     def forward(ctx, input):
#         ctx.save_for_backward(input)
#         out = torch.zeros_like(input)
#         out[input > 0] = 1.0
#         return out

#     @staticmethod
#     def backward(ctx, grad_output):
#         input, = ctx.saved_tensors
#         grad_input = grad_output.clone()
#         grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
#         return grad


# # ================================================================
# # DSNN (same interface as training)
# # ================================================================
# class DSNN(nn.Module):
#     def __init__(self, architecture, seed, alpha, beta, weight_scale,
#                  batch_size, threshold, simulation_time, learning_rate,
#                  reset_potential=0):
#         super().__init__()
#         self.architecture = architecture
#         self.simulation_time = simulation_time
#         self.batch_size = batch_size
#         self.threshold = threshold
#         self.reset_potential = reset_potential
#         self.alpha = alpha
#         self.beta = beta

#         # Weight init seed (will be overwritten by load_state_dict, but kept for compatibility)
#         torch.manual_seed(seed)

#         self.weights = nn.ParameterList()
#         for i in range(len(architecture) - 1):
#             w = torch.Tensor(architecture[i], architecture[i + 1])
#             nn.init.normal_(w, mean=0.0, std=weight_scale / np.sqrt(architecture[i]))
#             self.weights.append(nn.Parameter(w))

#         self.spike_fn = SurrGradSpike.apply
#         # Optimizer not used in eval, but kept for interface compatibility
#         self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

#     def forward(self, x):
#         batch_size = x.size(0)
#         syn, mem, spk = [], [], []

#         for l in range(len(self.weights)):
#             syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
#             mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
#             spk.append([])

#         mem_rec = []
#         all_spikes = []
#         ac_count = 0              # synaptic fan-out additions
#         internal_mac_count = 0    # internal state update MAC-like ops

#         for t in range(self.simulation_time):
#             inp = x[:, t, :]
#             layer_spikes = []

#             for l in range(len(self.weights)):
#                 # Synaptic operations
#                 if l == 0:
#                     h = torch.mm(inp, self.weights[l])
#                     num_spikes = torch.sum(inp > 0).item()
#                 else:
#                     h = torch.mm(spk[l - 1][-1], self.weights[l])
#                     num_spikes = torch.sum(spk[l - 1][-1]).item()

#                 ac_count += num_spikes * self.weights[l].size(1)

#                 # Internal state updates:
#                 # syn = alpha * syn + h      (1 mult + 1 add)
#                 # mem = beta * mem + syn     (1 mult + 1 add)
#                 num_neurons = self.weights[l].size(1)
#                 internal_mac_count += num_neurons * 2   # alpha*syn + h
#                 internal_mac_count += num_neurons * 2   # beta*mem + syn

#                 syn[l] = self.alpha * syn[l] + h
#                 mem[l] = self.beta * mem[l] + syn[l]

#                 if l < len(self.weights) - 1:
#                     mthr = mem[l] - self.threshold
#                     spk_current = self.spike_fn(mthr)

#                     # Approx. 2 extra MACs per spiking neuron for reset
#                     internal_mac_count += num_neurons * 2 * spk_current.float().mean().item()

#                     mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
#                     spk[l].append(spk_current)
#                     layer_spikes.append(spk_current)
#                 else:
#                     # Output layer: no spikes, continuous Q-values
#                     layer_spikes.append(torch.zeros_like(mem[l]))

#                 if l == len(self.weights) - 1:
#                     mem_rec.append(mem[l])

#             all_spikes.append(layer_spikes)

#         q_values = mem[-1]
#         return q_values, mem_rec, all_spikes, ac_count, internal_mac_count


# # ================================================================
# # FIXED SDR PROJECTION (ALWAYS TRAINING SEED = 42)
# # ================================================================
# def get_fixed_sdr_projection():
#     """
#     Build the SDR projection matrix exactly as in training:
#         np.random.seed(42)
#         np.random.randn(27, 100) * 0.1

#     Using a local RandomState so that global np RNG is not affected.
#     """
#     rng = np.random.RandomState(42)
#     return rng.randn(27, 100) * 0.1


# # ================================================================
# # DSQN with SDR encoding (using FIXED projection)
# # ================================================================
# class DSQN:
#     def __init__(self, discount_factor=0.95, epsilon=0.1, e_min=1000, e_max=100000,
#                  dsnn_config=None, sdr_projection=None):
#         self.gamma = discount_factor
#         self.epsilon = epsilon
#         self.e_min = e_min
#         self.exp_replay = None

#         self.batch_size = dsnn_config['batch_size']
#         self.simulation_time = dsnn_config['simulation_time']
#         self.sdr_size = 100           # SDR dimensionality
#         self.sparsity = 0.1           # 10% active bits

#         # FIXED projection matrix (must match training)
#         if sdr_projection is None:
#             raise ValueError("sdr_projection must be provided and must match training seed 42.")
#         self.projection_matrix = sdr_projection.copy()  # shape (27, 100)

#         # Networks (target net kept for compatibility, even if unused in eval)
#         self.training_net = DSNN(**dsnn_config).to(device)
#         self.target_net = DSNN(**dsnn_config).to(device)

#     def population_encode(self, state):
#         """
#         SDR encoding:
#         - One-hot (9 cells × 3 states = 27)
#         - Linear projection with fixed matrix
#         - Top-k sparsification
#         - Add small temporal noise
#         """
#         encoded = torch.zeros(1, self.simulation_time, self.sdr_size, device=device)

#         # One-hot representation
#         one_hot_state = np.zeros(9 * 3)
#         for i, val in enumerate(state):
#             if val == 1:
#                 one_hot_state[i * 3 + 0] = 1  # X
#             elif val == -1:
#                 one_hot_state[i * 3 + 1] = 1  # O
#             else:
#                 one_hot_state[i * 3 + 2] = 1  # Empty

#         # Project to SDR space (fixed matrix)
#         sdr = np.dot(one_hot_state, self.projection_matrix)

#         # Keep top-k entries
#         k = int(self.sdr_size * self.sparsity)
#         active_indices = np.argpartition(sdr, -k)[-k:]
#         sdr_binary = np.zeros(self.sdr_size)
#         sdr_binary[active_indices] = 1.0

#         # Repeat over time with small noise
#         for t in range(self.simulation_time):
#             noise = np.random.normal(0, 0.01, self.sdr_size)
#             noisy_sdr = np.clip(sdr_binary + noise, 0, 1)
#             encoded[0, t] = torch.tensor(noisy_sdr, device=device)

#         return encoded

#     def observe(self, state, action_space=None):
#         """
#         Run DSNN forward, return chosen action + analysis info.
#         """
#         with torch.no_grad():
#             encoded_state = self.population_encode(state)
#             q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)

#             q_values = q_values.cpu().numpy().flatten()
#             if action_space is not None:
#                 valid_q = [(q_values[a], a) for a in action_space]
#                 action = max(valid_q, key=lambda x: x[0])[1]
#             else:
#                 action = np.argmax(q_values)

#             return action, mem_rec, all_spikes, ac_count, internal_mac_count

#     def load_model(self, filename):
#         """
#         Load trained SDR DSQN model.
#         """
#         try:
#             checkpoint = torch.load(filename, map_location=device)
#             self.training_net.load_state_dict(checkpoint['training_net'])
#             self.target_net.load_state_dict(checkpoint['target_net'])
#             self.epsilon = checkpoint.get('epsilon', self.epsilon)

#             self.training_net.eval()
#             self.target_net.eval()
#             print(f"DSQN model loaded from {filename}")
#         except FileNotFoundError:
#             raise FileNotFoundError(f"DSQN model file {filename} not found")


# # ================================================================
# # Analysis functions
# # ================================================================
# def analyze_decision_stabilization(mem_rec, simulation_time):
#     decision_times = []
#     final_decision = torch.argmax(mem_rec[-1], dim=1)
#     for b in range(mem_rec[0].size(0)):
#         stabilized = False
#         for t in range(simulation_time):
#             current_decision = torch.argmax(mem_rec[t], dim=1)[b]
#             if current_decision == final_decision[b]:
#                 stable = True
#                 for t_next in range(t, simulation_time):
#                     if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
#                         stable = False
#                         break
#                 if stable:
#                     decision_times.append(t + 1)
#                     stabilized = True
#                     break
#         if not stabilized:
#             decision_times.append(simulation_time)
#     return decision_times


# def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
#     total_spikes = 0
#     total_neurons = sum(architecture[1:-1])  # hidden layers only
#     spike_counts = []

#     for t in range(len(all_spikes)):
#         for l in range(len(all_spikes[t])):
#             if l < len(architecture) - 1:  # exclude output layer
#                 spikes = all_spikes[t][l]
#                 spike_count = torch.sum(spikes).item()
#                 total_spikes += spike_count
#                 spike_counts.append(spike_count)

#     sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
#     return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count


# # ================================================================
# # Utility functions: board, game logic
# # ================================================================
# def print_board(board, move=None, agent=None):
#     print("\nBoard state:")
#     symbols = {1: 'X', -1: 'O', 0: '_'}
#     grid_board = [['_' for _ in range(3)] for _ in range(3)]
#     for i, val in enumerate(board):
#         row, col = i // 3, i % 3
#         grid_board[row][col] = symbols[val]
#     for row in grid_board:
#         print(" | ".join(row))
#         print("-" * 9)
#     if move and agent:
#         print(f"{agent} played at position ({move[0]}, {move[1]})")


# def game_over(board):
#     winning_positions = [
#         [0, 1, 2], [3, 4, 5], [6, 7, 8],
#         [0, 3, 6], [1, 4, 7], [2, 5, 8],
#         [0, 4, 8], [2, 4, 6]
#     ]
#     for pos in winning_positions:
#         if board[pos[0]] == board[pos[1]] == board[pos[2]] != 0:
#             return True, 1 if board[pos[0]] == 1 else -1
#     if 0 not in board:
#         return True, 0
#     return False, None


# def random_move(board):
#     action_space = [i for i, val in enumerate(board) if val == 0]
#     return random.choice(action_space)


# # ================================================================
# # Play one game
# # ================================================================
# def play_game(agent, agent_first=True, visualize=False, collect_analysis=False):
#     flat_board = [0] * 9
#     grid_board = [['_' for _ in range(3)] for _ in range(3)]
#     current_player = 'x'
#     agent_symbol = 'X'
#     random_symbol = 'O'

#     analysis_data = {
#         'total_spikes': [],
#         'sparsity': [],
#         'decision_times': [],
#         'ac_counts': [],
#         'mac_counts': [],              # kept for compatibility (not used)
#         'internal_mac_counts': []
#     } if collect_analysis else None

#     if visualize:
#         print("\n=== New Game ===")
#         print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
#         print_board(flat_board)

#     while True:
#         if current_player == 'x':
#             # Agent plays X
#             action_space = [i for i, val in enumerate(flat_board) if val == 0]
#             if collect_analysis:
#                 action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(flat_board, action_space)
#                 total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(
#                     all_spikes, agent.training_net.architecture, ac_count, internal_mac_count
#                 )
#                 analysis_data['total_spikes'].append(total_spikes)
#                 analysis_data['sparsity'].append(sparsity)
#                 analysis_data['ac_counts'].append(ac_count)
#                 analysis_data['internal_mac_counts'].append(internal_mac_count)
#                 decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
#                 analysis_data['decision_times'].extend(decision_times)
#             else:
#                 action = agent.observe(flat_board, action_space)[0]

#             flat_board[action] = 1
#             row, col = action // 3, action % 3
#             grid_board[row][col] = 'x'
#             if visualize:
#                 print_board(flat_board, (row, col), f"Agent ({agent_symbol})")
#         else:
#             # Random plays O
#             action = random_move(flat_board)
#             flat_board[action] = -1
#             row, col = action // 3, action % 3
#             grid_board[row][col] = 'o'
#             if visualize:
#                 print_board(flat_board, (row, col), f"Random ({random_symbol})")

#         done, result = game_over(flat_board)
#         if done:
#             if visualize:
#                 if result == 0:
#                     print("Game ended in a draw!")
#                 elif result == 1:
#                     print(f"Agent ({agent_symbol}) wins!")
#                 else:
#                     print(f"Random ({random_symbol}) wins!")
#             if result == 0:
#                 return 'draw', analysis_data
#             elif result == 1:
#                 return 'agent', analysis_data
#             else:
#                 return 'random', analysis_data

#         current_player = 'o' if current_player == 'x' else 'x'


# # ================================================================
# # Single-seed evaluation
# # ================================================================
# def test_agent_vs_random(agent, agent_name, num_games=100, visualize_all=False):
#     results = {'agent': 0, 'random': 0, 'draw': 0}
#     analysis_data = {
#         'total_spikes': [],
#         'sparsity': [],
#         'decision_times': [],
#         'ac_counts': [],
#         'mac_counts': [],
#         'internal_mac_counts': []
#     }

#     agent.load_model("../saved_models/sdr_ttt.pth")

#     for i in range(num_games):
#         visualize = visualize_all
#         winner, game_analysis = play_game(agent, agent_first=True,
#                                           visualize=visualize,
#                                           collect_analysis=True)
#         results[winner] += 1
#         if game_analysis:
#             analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
#             analysis_data['sparsity'].extend(game_analysis['sparsity'])
#             analysis_data['decision_times'].extend(game_analysis['decision_times'])
#             analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
#             analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

#     # Print game results
#     print(f"\n{agent_name} vs. Random Agent - Final Results (Agent as X)")
#     print("=" * 50)
#     print(f"Wins ({agent_name}): {results['agent']}, "
#           f"Losses: {results['random']}, Draws: {results['draw']}")
#     print(f"Win rate:  {results['agent'] / num_games:.2%}")
#     print(f"Loss rate: {results['random'] / num_games:.2%}")
#     print(f"Draw rate: {results['draw'] / num_games:.2%}")

#     # Print analysis results
#     if analysis_data['total_spikes']:
#         print(f"\nDSQN (SDR Encoding) Analysis Results")
#         print("=" * 50)
#         print(f"Average Total Spikes per Decision: "
#               f"{float(np.mean(analysis_data['total_spikes'])):.2f} "
#               f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
#         print(f"Average Sparsity per Decision: "
#               f"{float(np.mean(analysis_data['sparsity'])):.2%} "
#               f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
#         print(f"Average ACs per Decision (spike-triggered additions): "
#               f"{float(np.mean(analysis_data['ac_counts'])):.2f} "
#               f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
#         print(f"Average Internal State Update MACs per Decision: "
#               f"{float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
#               f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")

#         decision_time_counts = np.bincount(
#             analysis_data['decision_times'],
#             minlength=agent.simulation_time + 1
#         )[1:]
#         print(f"Decision Stabilization Times (over all decisions):")
#         for t in range(agent.simulation_time):
#             print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
#                   f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")

#         # Energy metrics
#         total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
#         total_neurons = 256  # 128 + 128 hidden
#         total_possible_spikes = total_neurons * agent.simulation_time
#         f_r = total_spikes_avg / total_possible_spikes

#         avg_ac = float(np.mean(analysis_data['ac_counts']))
#         avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))

#         ann_energy = 18688 * 31         # ANN MACs * E_MAC
#         snn_energy = avg_ac + avg_internal_mac * 31

#         energy_ratio_detailed = snn_energy / ann_energy
#         energy_savings_detailed = (1 - energy_ratio_detailed) * 100

#         energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
#         energy_savings_simplified = (1 - energy_ratio_simplified) * 100

#         print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): "
#               f"{energy_ratio_simplified:.4f}")
#         print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
#         print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): "
#               f"{energy_ratio_detailed:.4f}")
#         print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")

#     return results, analysis_data


# # ================================================================
# # MAIN
# # ================================================================
# if __name__ == "__main__":
#     # >>> Change ONLY this seed to 42, 52, 62, ... for different runs <<<
#     GLOBAL_EVAL_SEED = 42

#     # Set global RNG seeds for evaluation randomness (opponent, noise)
#     random.seed(GLOBAL_EVAL_SEED)
#     np.random.seed(GLOBAL_EVAL_SEED)
#     torch.manual_seed(GLOBAL_EVAL_SEED)

#     # DSQN configuration (MUST keep seed=42 for compatibility with training)
#     dsnn_config = {
#         'architecture': [100, 128, 128, 9],
#         'seed': 42,          # training seed for DSNN weights (do NOT change)
#         'alpha': 0.9,
#         'beta': 0.85,
#         'weight_scale': 0.15,
#         'batch_size': 32,
#         'threshold': 0.1,
#         'simulation_time': 5,
#         'learning_rate': 0.0001,
#         'reset_potential': 0.0
#     }

#     # Fixed SDR projection matrix using training seed 42
#     fixed_projection = get_fixed_sdr_projection()

#     # Initialize DSQN
#     dsqn_agent = DSQN(dsnn_config=dsnn_config,
#                       sdr_projection=fixed_projection)

#     print("\nTesting DSQN (SDR Encoding, fixed SDR projection) vs. Random Agent (Agent as X)")
#     test_agent_vs_random(dsqn_agent, "DSQN", num_games=100, visualize_all=False)


In [ ]:
##Evaluation for SDR encoding method by stabilizing the matrix projection.  

import random
import numpy as np
import torch
import torch.nn as nn
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==============================================================
# DSNN + Surrogate Gradient
# ==============================================================
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale,
                 threshold, simulation_time, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta

        torch.manual_seed(seed)
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.empty(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))

        self.spike_fn = SurrGradSpike.apply

    def forward(self, x):
        B = x.size(0)
        syn = [torch.zeros(B, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(B, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]

        mem_rec = []
        all_spikes = []
        ac_count = 0
        internal_mac_count = 0

        for t in range(self.simulation_time):
            inp = x[:, t, :]
            layer_spikes = []

            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.mm(inp, self.weights[l])
                    num_spikes = (inp > 0).sum().item()
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                    num_spikes = spk[l-1][-1].sum().item()

                ac_count += num_spikes * self.weights[l].size(1)
                neurons = self.weights[l].size(1)
                internal_mac_count += neurons * 2

                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]

                if l < len(self.weights)-1:
                    spk_cur = self.spike_fn(mem[l] - self.threshold)
                    internal_mac_count += neurons * 2 * spk_cur.float().mean().item()
                    mem[l] = mem[l] * (1 - spk_cur) + self.reset_potential * spk_cur
                    spk[l].append(spk_cur)
                    layer_spikes.append(spk_cur)
                else:
                    layer_spikes.append(torch.zeros_like(mem[l]))

                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])

            all_spikes.append(layer_spikes)

        return mem[-1], mem_rec, all_spikes, ac_count, internal_mac_count

# ==============================================================
# DSQN with SDR Encoding
# ==============================================================
class DSQN:
    def __init__(self, dsnn_config):
        self.simulation_time = dsnn_config['simulation_time']
        self.sdr_size = 100
        self.sparsity = 0.1
        self.seed = dsnn_config['seed']
        np.random.seed(self.seed)
        self.projection = np.random.randn(27, self.sdr_size) * 0.1  # 9 cells × 3 states → SDR

        self.training_net = DSNN(**dsnn_config).to(device)

    def sdr_encode(self, state):
        one_hot = np.zeros(27)
        for i, val in enumerate(state):
            if val == 1:
                one_hot[i*3 + 0] = 1
            elif val == -1:
                one_hot[i*3 + 1] = 1
            else:
                one_hot[i*3 + 2] = 1

        projection = np.dot(one_hot, self.projection)
        k = int(self.sdr_size * self.sparsity)
        active = np.argpartition(projection, -k)[-k:]
        sdr = np.zeros(self.sdr_size)
        sdr[active] = 1.0

        encoded = torch.zeros(1, self.simulation_time, self.sdr_size, device=device)
        for t in range(self.simulation_time):
            noise = np.random.normal(0, 0.01, self.sdr_size)
            noisy = np.clip(sdr + noise, 0, 1)
            encoded[0, t] = torch.tensor(noisy, device=device)
        return encoded

    def observe(self, state, action_space=None):
        with torch.no_grad():
            x = self.sdr_encode(state)
            q, mem_rec, spikes, ac, imac = self.training_net(x)
            q = q.cpu().numpy().flatten()
            action = max((q[a], a) for a in action_space)[1] if action_space else np.argmax(q)
            return action, mem_rec, spikes, ac, imac

    def load_model(self, path):
        ckpt = torch.load(path, map_location=device)
        self.training_net.load_state_dict(ckpt['training_net'])
        self.training_net.eval()

# ==============================================================
# Analysis & Game Logic
# ==============================================================
def analyze_decision_stabilization(mem_rec, T):
    times = []
    final = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        for t in range(T):
            if torch.argmax(mem_rec[t], dim=1)[b] == final[b]:
                if all(torch.argmax(mem_rec[k], dim=1)[b] == final[b] for k in range(t, T)):
                    times.append(t+1)
                    break
        else:
            times.append(T)
    return times

def analyze_spikes(all_spikes, arch, ac_count, internal_mac_count):
    total_spikes = sum(s.sum().item() for t in all_spikes for l, s in enumerate(t) if l < len(arch)-2)
    total_neurons = sum(arch[1:-1])
    sparsity = 1 - total_spikes / (total_neurons * len(all_spikes))
    return total_spikes, sparsity, ac_count, internal_mac_count

def game_over(board):
    lines = [[0,1,2],[3,4,5],[6,7,8],[0,3,6],[1,4,7],[2,5,8],[0,4,8],[2,4,6]]
    for a,b,c in lines:
        if board[a] == board[b] == board[c] != 0:
            return True, 1 if board[a] == 1 else -1
    return (True, 0) if 0 not in board else (False, None)

def random_move(board):
    return random.choice([i for i,v in enumerate(board) if v == 0])

def play_game(agent, collect_analysis=False):
    board = [0]*9
    data = {'total_spikes':[], 'sparsity':[], 'decision_times':[], 
            'ac_counts':[], 'internal_mac_counts':[]} if collect_analysis else None

    while True:
        empty = sum(1 for x in board if x == 0)
        if empty % 2 == 1:  # agent first
            action_space = [i for i,v in enumerate(board) if v == 0]
            action, mem_rec, spikes, ac, imac = agent.observe(board, action_space)
            if collect_analysis:
                ts, sp, ac, imac = analyze_spikes(spikes, agent.training_net.architecture, ac, imac)
                data['total_spikes'].append(ts)
                data['sparsity'].append(sp)
                data['ac_counts'].append(ac)
                data['internal_mac_counts'].append(imac)
                data['decision_times'].extend(analyze_decision_stabilization(mem_rec, agent.simulation_time))
            board[action] = 1
        else:
            board[random_move(board)] = -1

        done, res = game_over(board)
        if done:
            winner = 'agent' if res == 1 else ('draw' if res == 0 else 'random')
            return winner, data

# ==============================================================
# FULL MULTI-SEED EVALUATION FOR SDR ENCODING
# ==============================================================
def evaluate_sdr_encoding(seeds=[42, 52, 62, 72, 82], games_per_seed=500):
    print("="*80)
    print("DSQN (SDR Encoding) – Full Evaluation over 5 Seeds")
    print("Using fixed trained model: ../saved_models/sdr_ttt.pth")
    print("="*80)

    config = {
        'architecture': [100, 128, 128, 9],
        'seed': 42,                      # only for training
        'alpha': 0.9, 'beta': 0.85,
        'weight_scale': 0.15,
        'threshold': 0.1,
        'simulation_time': 5,
        'reset_potential': 0.0
    }

    ANN_MACS = 18688
    E_RATIO = 31

    results = []
    metrics = []

    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        agent = DSQN(config)
        agent.load_model("../saved_models/sdr_ttt.pth")

        wins = draws = losses = 0
        per_decision = {
            'spikes':[], 'sparsity':[], 'ac':[], 'internal_mac':[], 'stab':[]
        }

        for _ in range(games_per_seed):
            winner, data = play_game(agent, collect_analysis=True)
            wins   += winner == 'agent'
            draws  += winner == 'draw'
            losses += winner == 'random'

            if data:
                per_decision['spikes'].extend(data['total_spikes'])
                per_decision['sparsity'].extend(data['sparsity'])
                per_decision['ac'].extend(data['ac_counts'])
                per_decision['internal_mac'].extend(data['internal_mac_counts'])
                per_decision['stab'].extend(data['decision_times'])

        win_rate = wins / games_per_seed
        non_loss = (wins + draws) / games_per_seed
        loss_rate = losses / games_per_seed

        print(f"\nSeed {seed:2d} → Wins:{wins:3d}  Draws:{draws:3d}  Losses:{losses:3d}")
        print(f"   Win %: {win_rate:.2%} | Win+Draw %: {non_loss:.2%} | Loss %: {loss_rate:.2%}")

        avg_spikes = np.mean(per_decision['spikes'])
        avg_sparsity = np.mean(per_decision['sparsity'])
        avg_ac = np.mean(per_decision['ac'])
        avg_internal_mac = np.mean(per_decision['internal_mac'])

        f_r = avg_spikes / (256 * 5)
        simplified_savings = (1 - (5 * f_r / E_RATIO)) * 100
        detailed_savings = (1 - (avg_ac + avg_internal_mac * E_RATIO) / (ANN_MACS * E_RATIO)) * 100

        hist = np.bincount(per_decision['stab'], minlength=6)[1:6]

        print(f"  Avg Spikes: {avg_spikes:.1f} | ACs: {avg_ac:,.0f} | Int.MACs: {avg_internal_mac:.1f}")
        print(f"  Simplified Savings: {simplified_savings:.1f}% | Detailed Savings: {detailed_savings:.1f}%")
        print(f"  Stabilization → " + "  ".join([f"T{t+1}:{c}" for t,c in enumerate(hist)]))

        results.append({'win':win_rate, 'non_loss':non_loss, 'loss':loss_rate})
        metrics.append({
            'spikes':avg_spikes, 'sparsity':avg_sparsity,
            'ac':avg_ac, 'internal_mac':avg_internal_mac,
            'simplified':simplified_savings, 'detailed':detailed_savings,
            'hist':hist.copy()
        })

    # Final Summary
    print("\n" + "="*80)
    print("FINAL RESULTS – SDR Encoding (Mean ± Std over 5 seeds)")
    print("="*80)

    print(f"Win Rate          : {np.mean([r['win'] for r in results]):.2%} ± {np.std([r['win'] for r in results]):.2%}")
    print(f"Win + Draw Rate   : {np.mean([r['non_loss'] for r in results]):.2%} ± {np.std([r['non_loss'] for r in results]):.2%}")
    print(f"Loss Rate         : {np.mean([r['loss'] for r in results]):.2%} ± {np.std([r['loss'] for r in results]):.2%}")

    print("\nEfficiency Metrics (per decision):")
    print(f"Avg Spikes        : {np.mean([m['spikes'] for m in metrics]):.1f} ± {np.std([m['spikes'] for m in metrics]):.1f}")
    print(f"Avg Sparsity      : {np.mean([m['sparsity'] for m in metrics]):.2%} ± {np.std([m['sparsity'] for m in metrics]):.2%}")
    print(f"Avg ACs           : {np.mean([m['ac'] for m in metrics]):,.0f} ± {np.std([m['ac'] for m in metrics]):,.0f}")
    print(f"Avg Internal MACs : {np.mean([m['internal_mac'] for m in metrics]):.1f} ± {np.std([m['internal_mac'] for m in metrics]):.1f}")
    print(f"Simplified Savings: {np.mean([m['simplified'] for m in metrics]):.1f}% ± {np.std([m['simplified'] for m in metrics]):.1f}%")
    print(f"Detailed Savings  : {np.mean([m['detailed'] for m in metrics]):.1f}% ± {np.std([m['detailed'] for m in metrics]):.1f}%")

    avg_hist = np.mean([m['hist'] for m in metrics], axis=0)
    total = avg_hist.sum()
    print("\nDecision Stabilization Time (averaged):")
    for t, c in enumerate(avg_hist, 1):
        print(f"  Time Step {t}: {c:6.1f} decisions ({c/total:.1%})")

if __name__ == "__main__":
    evaluate_sdr_encoding(seeds=[42, 52, 62, 72, 82], games_per_seed=100)

DSQN (SDR Encoding) – Full Evaluation over 5 Seeds
Using fixed trained model: ../saved_models/sdr_ttt.pth

Seed 42 → Wins: 86  Draws: 14  Losses:  0
   Win %: 86.00% | Win+Draw %: 100.00% | Loss %: 0.00%
  Avg Spikes: 211.8 | ACs: 48,542 | Int.MACs: 3073.6
  Simplified Savings: 97.3% | Detailed Savings: 75.2%
  Stabilization → T1:37  T2:245  T3:59  T4:58  T5:38

Seed 52 → Wins: 72  Draws: 28  Losses:  0
   Win %: 72.00% | Win+Draw %: 100.00% | Loss %: 0.00%
  Avg Spikes: 213.3 | ACs: 48,611 | Int.MACs: 3076.6
  Simplified Savings: 97.3% | Detailed Savings: 75.1%
  Stabilization → T1:57  T2:232  T3:63  T4:64  T5:36

Seed 62 → Wins: 83  Draws: 17  Losses:  0
   Win %: 83.00% | Win+Draw %: 100.00% | Loss %: 0.00%
  Avg Spikes: 212.4 | ACs: 48,649 | Int.MACs: 3074.8
  Simplified Savings: 97.3% | Detailed Savings: 75.1%
  Stabilization → T1:38  T2:258  T3:45  T4:48  T5:38

Seed 72 → Wins: 85  Draws: 15  Losses:  0
   Win %: 85.00% | Win+Draw %: 100.00% | Loss %: 0.00%
  Avg Spikes: 211.3 | 